# Pest Detection + Hybrid Classification Pipeline

YOLOv8 (Single Class: Pest) + HybridPestNet (EfficientNetB3)

Author: Parampreet Singh

Generated on: 2026-02-13

In [ ]:
!pip install ultralytics -q

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import time

from ultralytics import YOLO
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input
from sklearn.metrics import confusion_matrix, classification_report

## Load Models

In [ ]:
# Load YOLO Model
yolo_model = YOLO('/kaggle/input/yolo_model/best.pt')

# Load Hybrid Classifier
classifier = load_model('/kaggle/input/hybrid_model/best_pestnet.keras')

## Detection + Classification Pipeline

In [ ]:
IMG_SIZE = (300, 300)

def detect_and_classify(image_path):
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    results = yolo_model(image_rgb)

    if len(results[0].boxes) == 0:
        return {'Detection_Status': 'No Pest Found'}

    box = results[0].boxes.xyxy[0].cpu().numpy()
    x1, y1, x2, y2 = map(int, box)

    crop = image_rgb[y1:y2, x1:x2]
    crop = cv2.resize(crop, IMG_SIZE)

    crop = preprocess_input(crop)
    crop = np.expand_dims(crop, axis=0)

    preds = classifier.predict(crop)

    confidence = float(np.max(preds))
    pred_class = int(np.argmax(preds))

    return {
        'Detection_Status': 'Pest Found',
        'Pest_Class': pred_class,
        'Confidence': confidence
    }

## Test Pipeline

In [ ]:
# Example Test
result = detect_and_classify('/kaggle/input/sample-image/sample.jpg')
print(result)

## Inference Speed Measurement

In [ ]:
start = time.time()
_ = detect_and_classify('/kaggle/input/sample-image/sample.jpg')
end = time.time()

print('Inference Time:', end-start, 'seconds')

## Training Evaluation Section (Use After Training)

Plot accuracy, loss, confusion matrix and classification report.

In [ ]:
# Example plotting code
def plot_training(history):
    plt.figure(figsize=(8,6))
    plt.plot(history.history['accuracy'], label='Train Acc')
    plt.plot(history.history['val_accuracy'], label='Val Acc')
    plt.legend(); plt.title('Accuracy'); plt.show()

    plt.figure(figsize=(8,6))
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.legend(); plt.title('Loss'); plt.show()